<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/o1_Arul_reactive_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Reactive Agents with LangChain and LLM Reasoning

## What you will learn

By the end, you should be able to:
- Explain a reactive agent as a stimulus-response system.
- Use LangChain tools to represent actions.
- Add an LLM reasoning layer without making the agent slow or unpredictable.
- Keep the final action deterministic when the use case requires speed.

This notebook uses LangChain with `ChatOpenAI` for real LLM reasoning. Set `OPENAI_API_KEY` before running the LLM cells.

In [ ]:
import os
import re
from operator import itemgetter
from google.colab import userdata # Import userdata to access Colab secrets

# Install langchain_openai if not already installed
try:
    from langchain_openai import ChatOpenAI
except ImportError:
    %pip install -U langchain-openai
    from langchain_openai import ChatOpenAI

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.tools import tool


def make_llm(temperature=0):
    """Create a real OpenAI-backed chat model for the notebook examples."""
    # Get the API key from Colab's secrets
    openai_api_key = userdata.get("OPENAI_API_KEY")
    if not openai_api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY is not set in Colab secrets. Set it before running LLM cells so the notebook uses a real model."
        )
    # Set it as an environment variable for ChatOpenAI
    os.environ["OPENAI_API_KEY"] = openai_api_key

    return ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        temperature=temperature,
    )


def print_box(title, text):
    print(f"\n{title}\n" + "-" * len(title))
    print(text)


print("LangChain setup ready.")
print("LLM provider: OpenAI via ChatOpenAI")
print("Model:", os.getenv("OPENAI_MODEL", "gpt-4o-mini"))
print("OPENAI_API_KEY configured:", bool(userdata.get("OPENAI_API_KEY")))

LangChain setup ready.
LLM provider: OpenAI via ChatOpenAI
Model: gpt-4o-mini
OPENAI_API_KEY configured: True


## 1. Define Action Tools

A reactive agent usually chooses from a small set of predefined actions.

In LangChain, tools are a natural way to represent those actions.

In [ ]:
@tool
def turn_heater_on() -> str:
    """Turn on the heater when the room is too cold."""
    return "Heater turned on"


@tool
def turn_cooling_on() -> str:
    """Turn on cooling when the room is too hot."""
    return "Cooling turned on"


@tool
def do_nothing() -> str:
    """Keep the system state unchanged."""
    return "No action taken"


tools = {
    "turn_heater_on": turn_heater_on,
    "turn_cooling_on": turn_cooling_on,
    "do_nothing": do_nothing,
}

print("Available tools:", list(tools))

Available tools: ['turn_heater_on', 'turn_cooling_on', 'do_nothing']


## 2. Add a Fast Reactive Policy

The policy does not ask the LLM to decide. It maps the current percept directly to an action.

This keeps the agent fast and predictable.

In [ ]:
def reactive_policy(percept):
    temperature = percept["temperature_celsius"]
    if temperature < 20:
        return "turn_heater_on"
    if temperature > 25:
        return "turn_cooling_on"
    return "do_nothing"


policy_runnable = RunnableLambda(lambda percept: {"percept": percept, "action": reactive_policy(percept)})

test_percepts = [
    {"room": "A", "temperature_celsius": 18},
    {"room": "B", "temperature_celsius": 23},
    {"room": "C", "temperature_celsius": 29},
]

for percept in test_percepts:
    print(policy_runnable.invoke(percept))

{'percept': {'room': 'A', 'temperature_celsius': 18}, 'action': 'turn_heater_on'}
{'percept': {'room': 'B', 'temperature_celsius': 23}, 'action': 'do_nothing'}
{'percept': {'room': 'C', 'temperature_celsius': 29}, 'action': 'turn_cooling_on'}


## 3. Use the LLM for Reasoning, Not Control

Here the LLM explains why the rule fired. This is useful for transparency during debugging or audits.

The important design choice: the LLM explains the action, but the deterministic rule still selects the action.

In [ ]:
llm = make_llm()

explanation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain reactive agent decisions in one concise sentence."),
    ("human", "Percept: {percept}\nSelected action: {action}\nExplain why this action follows the rule."),
])

explanation_chain = explanation_prompt | llm | StrOutputParser()

for percept in test_percepts:
    decision = policy_runnable.invoke(percept)
    explanation = explanation_chain.invoke(decision)
    print_box(f"{percept}", f"Action: {decision['action']}\nReasoning: {explanation}")


{'room': 'A', 'temperature_celsius': 18}
----------------------------------------
Action: turn_heater_on
Reasoning: The action to turn the heater on follows the rule because the current temperature in room A (18°C) is likely below the desired comfort level, prompting the agent to increase the temperature by activating the heater.

{'room': 'B', 'temperature_celsius': 23}
----------------------------------------
Action: do_nothing
Reasoning: The action "do_nothing" follows the rule because the current temperature of 23°C in room B is within a comfortable range, indicating no need for any adjustments or interventions.

{'room': 'C', 'temperature_celsius': 29}
----------------------------------------
Action: turn_cooling_on
Reasoning: The action to turn the cooling on follows the rule because the percept indicates that the temperature in room C is 29 degrees Celsius, which is likely above a predefined comfort threshold, prompting the agent to activate cooling to maintain a comfortable en

## 4. Execute the Selected Tool

After the policy chooses an action, the agent invokes the matching tool.

In [ ]:
def execute_decision(decision):
    tool_name = decision["action"]
    result = tools[tool_name].invoke({})
    return {**decision, "tool_result": result}


reactive_agent = policy_runnable | RunnableLambda(execute_decision)

for percept in test_percepts:
    print(reactive_agent.invoke(percept))

{'percept': {'room': 'A', 'temperature_celsius': 18}, 'action': 'turn_heater_on', 'tool_result': 'Heater turned on'}
{'percept': {'room': 'B', 'temperature_celsius': 23}, 'action': 'do_nothing', 'tool_result': 'No action taken'}
{'percept': {'room': 'C', 'temperature_celsius': 29}, 'action': 'turn_cooling_on', 'tool_result': 'Cooling turned on'}


## 5. Natural-Language Percepts

In many real systems, the input arrives as text. The LLM can convert text into a simple percept, then the reactive policy takes over.

In [ ]:
parser_llm = make_llm()

parse_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract only the numeric temperature in Celsius. Return only the number."),
    ("human", "{message}"),
])

parse_temperature = parse_prompt | parser_llm | StrOutputParser()

messages = [
    "Sensor says room A is chilly, approximately 18 Celsius.",
    "Room B is getting warm at around 27 Celsius.",
]

for message in messages:
    raw_temperature = parse_temperature.invoke({"message": message}).strip()
    match = re.search(r"-?\d+", raw_temperature)
    if not match:
        raise ValueError(f"Could not parse a temperature from: {raw_temperature!r}")
    temperature = int(match.group())
    percept = {"room": "unknown", "temperature_celsius": temperature}
    print(message)
    print(reactive_agent.invoke(percept))

Sensor says room A is chilly, approximately 18 Celsius.
{'percept': {'room': 'unknown', 'temperature_celsius': 18}, 'action': 'turn_heater_on', 'tool_result': 'Heater turned on'}
Room B is getting warm at around 27 Celsius.
{'percept': {'room': 'unknown', 'temperature_celsius': 27}, 'action': 'turn_cooling_on', 'tool_result': 'Cooling turned on'}


## Exercise

Add a new action called `send_alert` when temperature is below 10 or above 35.

Questions to answer:
- Should the LLM choose the alert action, or should the rule choose it?
- Why might safety-critical systems prefer deterministic thresholds?

In [ ]:
@tool
def send_alert() -> str:
    """Send an alert for unsafe temperature readings."""
    return "Safety alert sent"


tools["send_alert"] = send_alert


def safer_reactive_policy(percept):
    temperature = percept["temperature_celsius"]
    if temperature < 10 or temperature > 35:
        return "send_alert"
    return reactive_policy(percept)


safer_agent = (
    RunnableLambda(lambda percept: {"percept": percept, "action": safer_reactive_policy(percept)})
    | RunnableLambda(execute_decision)
)

print(safer_agent.invoke({"room": "lab", "temperature_celsius": 38}))

{'percept': {'room': 'lab', 'temperature_celsius': 38}, 'action': 'send_alert', 'tool_result': 'Safety alert sent'}


## Key Takeaway

Reactive agents are strongest when the action must be immediate and predictable. LangChain tools make actions explicit, while an LLM can help interpret natural language inputs or explain decisions.